# 🏥 Disease Prediction — Exploratory Data Analysis
> CodeAlpha ML Internship — Task 4

This notebook explores all three datasets before model training.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.data_loader import load_dataset
from src.features.feature_engineering import prepare_features

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('Setup complete ✅')

## 1. Heart Disease Dataset

In [ ]:
df_heart, target_heart = load_dataset('heart')
print(f'Shape: {df_heart.shape}')
df_heart.head()

In [ ]:
print('Missing values:')
print(df_heart.isnull().sum())
print(f'\nClass distribution:')
print(df_heart[target_heart].value_counts())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
for ax, col in zip(axes.flat, cols):
    sns.histplot(data=df_heart, x=col, hue=target_heart, ax=ax, kde=True, bins=20)
    ax.set_title(col)
plt.suptitle('Heart Disease Feature Distributions by Class', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
corr = df_heart.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, linewidths=0.5)
plt.title('Heart Disease Correlation Matrix')
plt.show()

## 2. Diabetes Dataset

In [ ]:
df_diab, target_diab = load_dataset('diabetes')
print(f'Shape: {df_diab.shape}')
df_diab.describe()

In [ ]:
# Pairplot for key diabetes features
key_cols = ['glucose', 'bmi', 'age', 'insulin', 'outcome']
df_plot = df_diab[key_cols].dropna()
df_plot['outcome'] = df_plot['outcome'].astype(str)
sns.pairplot(df_plot, hue='outcome', diag_kind='kde',
             plot_kws={'alpha': 0.4})
plt.suptitle('Diabetes Key Feature Pairplot', y=1.02)
plt.show()

## 3. Breast Cancer Dataset

In [ ]:
df_bc, target_bc = load_dataset('breast_cancer')
print(f'Shape: {df_bc.shape}')
df_bc.head()

In [ ]:
# Top correlated features with target
corr_with_target = df_bc.corr(numeric_only=True)[target_bc].drop(target_bc).abs().sort_values(ascending=False)
top_features = corr_with_target.head(10)

plt.figure(figsize=(8, 5))
top_features.plot(kind='barh', color='steelblue')
plt.title('Top 10 Features Correlated with Cancer Type')
plt.xlabel('|Correlation|')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Box plots for top features by class
top_cols = list(top_features.index[:6])
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
labels = {0: 'Malignant', 1: 'Benign'}
df_bc_plot = df_bc.copy()
df_bc_plot['diagnosis'] = df_bc_plot[target_bc].map(labels)

for ax, col in zip(axes.flat, top_cols):
    sns.boxplot(data=df_bc_plot, x='diagnosis', y=col, ax=ax,
                palette=['#DD8452', '#4C72B0'])
    ax.set_title(col, fontsize=10)

plt.suptitle('Breast Cancer Top Features by Diagnosis', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Feature Engineering Preview

In [ ]:
for disease in ['heart', 'diabetes', 'breast_cancer']:
    df, tc = load_dataset(disease)
    X, y, prep = prepare_features(disease, df, tc)
    print(f'{disease.upper()}: {X.shape[1]} features after engineering')
    print(f'  New cols: {[c for c in X.columns if c not in df.columns]}')
    print()